<a href="https://colab.research.google.com/github/shr-eexe/Multi-Asset-Financial-Directional-Forecasting-/blob/main/prediction2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: install libs (run once)
!pip install yfinance finnhub-python ta transformers torch torchvision torchaudio joblib pandas scikit-learn vaderSentiment --quiet


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.3 MB/s eta 0:00:00


In [2]:
# Cell 2: mount drive (run if you want to persist)
from google.colab import drive
drive.mount('/content/drive')
# set a folder in your Drive where files will be saved
ARTIFACT_DIR = "/content/drive/MyDrive/stock_artifacts"
import os
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print("ARTIFACT_DIR:", ARTIFACT_DIR)


Mounted at /content/drive
ARTIFACT_DIR: /content/drive/MyDrive/stock_artifacts


In [3]:
# Cell 3: set your API keys
FINNHUB_API_KEY = "d3pprn1r01qs89su9njgd3pprn1r01qs89su9nk0"   # or set to "" to skip fetching news
# If you use HuggingFace auth for private models, put token in HF_TOKEN (optional)
HF_TOKEN = ""


In [4]:
import os, time
from datetime import datetime, timedelta # FIX 1: Imports timedelta
import pandas as pd, numpy as np
import yfinance as yf
import joblib
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
import ta
warnings.filterwarnings('ignore')

# ---------- PARAMETERS ----------
TICKERS = ["AAPL","MSFT","GOOGL","AMZN"]
START_DATE = "2018-01-01"
# FIX 2: Set END_DATE to yesterday for reliable, complete data
END_DATE = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
LOOKBACK_WINDOW = 21
TEST_SIZE = 0.2

# ARTIFACT_DIR declared earlier (Drive or local)
try:
    ARTIFACT_DIR
except NameError:
    ARTIFACT_DIR = "/content/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---------- helpers (FIXED) ----------
def fetch_yf_price(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty:
        return df

    # Fix potential MultiIndex issues
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
        df = df.loc[:, ~df.columns.duplicated()]

    # Keep only necessary columns
    df = df[['Close', 'Open', 'High', 'Low', 'Volume']].copy()

    # 🔧 Flatten any 2D columns
    for col in df.columns:
        if hasattr(df[col].values, "shape") and len(df[col].values.shape) == 2:
            df[col] = df[col].values.reshape(-1)
        elif isinstance(df[col].iloc[0], (list, np.ndarray)):
            df[col] = df[col].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # FIX 3: Removed aggressive df.dropna() here to retain maximum rows
    return df


# Simple sentiment function using VADER
analyzer = SentimentIntensityAnalyzer()
def headline_sentiment_vader(headline):
    try:
        s = analyzer.polarity_scores(str(headline))
        return s['compound']
    except:
        return 0.0

# ---------- feature engineering (FINAL, ROBUST FIX) ----------
def add_technical_indicators(df):
    """
    Calculates returns, ensures all core data is numeric, and fills NaNs
    to guarantee a continuous time series before Cell 6 runs.
    """
    df = df.copy()

    # 1. Ensure core columns are numeric
    core_cols = ['Close', 'Open', 'High', 'Low', 'Volume']
    for col in core_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 2. Calculate returns BEFORE filling (to ensure clean returns values)
    df['returns'] = df['Close'].pct_change()

    # 3. Aggressively fill NaNs for a guaranteed full time series:
    # Fill remaining NaNs in core data with 0. This is the only way to ensure
    # the entire history remains intact when grouped later.
    df = df.fillna(0)

    # After fill, calculate returns again since the first row had a NaN/0
    # The redundant recalculation is necessary if any row was filled with 0.
    df['returns'] = df['Close'].pct_change().fillna(0)

    # Drop first row which always has NaN due to pct_change before the final fill
    df = df.iloc[1:].copy()

    return df

In [6]:
# ---------- Cell 5: Build or load dataset (fixed multi-ticker version) ----------
import yfinance as yf
import pandas as pd
import numpy as np
import os

ARTIFACT_DIR = "/content/drive/MyDrive/stock_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

DATA_PATH = os.path.join(ARTIFACT_DIR, "multi_stock_dataset.csv")

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN"]
START_DATE = "2018-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")


def fetch_yf_price(ticker):
    """Download single ticker data safely."""
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    if df.empty:
        print(f"⚠️ No data for {ticker}, skipping.")
        return None
    df = df.reset_index()
    df["ticker"] = ticker
    return df[["Date", "Open", "High", "Low", "Close", "Volume", "ticker"]]


# ---------------- BUILD DATASET ----------------
if not os.path.exists(DATA_PATH):
    all_dfs = []

    for t in TICKERS:
        print(f"Downloading: {t}")
        df_t = fetch_yf_price(t)
        if df_t is not None and not df_t.empty:
            print(f"✅ {t} done — {len(df_t)} rows")
            all_dfs.append(df_t)
        else:
            print(f"⚠️ Skipped {t} (no data)")

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df.rename(columns={"Date": "date"}, inplace=True)

    # --- Add returns + sentiment placeholder ---
    combined_df["returns"] = combined_df.groupby("ticker")["Close"].pct_change()
    combined_df["daily_sentiment"] = 0.0

    combined_df.dropna(subset=["returns"], inplace=True)
    combined_df.reset_index(drop=True, inplace=True)

    # Save clean version
    combined_df.to_csv(DATA_PATH, index=False)
    print(f"💾 Saved dataset to: {DATA_PATH}")
    print(f"✅ Dataset successfully built! Total rows: {len(combined_df)}")

else:
    print("📂 Loading existing dataset...")
    combined_df = pd.read_csv(DATA_PATH)

    # ---------- 🔧 FIX DATE CORRUPTION ----------
    if "date.1" in combined_df.columns:
        # Keep the correct 'date.1' column
        combined_df["date"] = pd.to_datetime(combined_df["date.1"], errors="coerce")
        combined_df.drop(columns=["date.1"], inplace=True)
    else:
        # Fix direct numeric or string date corruption
        combined_df["date"] = pd.to_datetime(combined_df["date"], errors="coerce")

    # Drop invalid / missing dates
    combined_df = combined_df.dropna(subset=["date"]).reset_index(drop=True)
    print("✅ Date column fixed and standardized.")
# ------------------------------------------------

# Quick preview
print("✅ Loaded tickers:", combined_df["ticker"].unique().tolist())
print("Rows:", len(combined_df))
print("Columns:", combined_df.columns.tolist())
print(combined_df.head())


Downloading: AAPL
✅ AAPL done — 1994 rows
Downloading: MSFT
✅ MSFT done — 1994 rows
Downloading: GOOGL
✅ GOOGL done — 1994 rows
Downloading: AMZN
✅ AMZN done — 1994 rows


ValueError: Cannot set a DataFrame with multiple columns to the single column returns

In [7]:
# ---------- Cell 6: Add technical indicators + target ----------
import pandas as pd
import numpy as np

def add_technical_indicators(df):
    df = df.copy()
    df = df.sort_values(["ticker", "date"])

    # --- Moving averages ---
    df["ma5"] = df.groupby("ticker")["Close"].transform(lambda x: x.rolling(5).mean())
    df["ma10"] = df.groupby("ticker")["Close"].transform(lambda x: x.rolling(10).mean())
    df["ma20"] = df.groupby("ticker")["Close"].transform(lambda x: x.rolling(20).mean())

    # --- Exponential moving averages ---
    df["ema10"] = df.groupby("ticker")["Close"].transform(lambda x: x.ewm(span=10, adjust=False).mean())
    df["ema20"] = df.groupby("ticker")["Close"].transform(lambda x: x.ewm(span=20, adjust=False).mean())

    # --- Momentum ---
    df["momentum"] = df.groupby("ticker")["Close"].transform(lambda x: x.diff(4))

    # --- Volatility (rolling std of returns) ---
    df["volatility"] = df.groupby("ticker")["returns"].transform(lambda x: x.rolling(10).std())

    # --- MACD + Signal line ---
    exp1 = df.groupby("ticker")["Close"].transform(lambda x: x.ewm(span=12, adjust=False).mean())
    exp2 = df.groupby("ticker")["Close"].transform(lambda x: x.ewm(span=26, adjust=False).mean())
    df["macd"] = exp1 - exp2
    df["macd_signal"] = df.groupby("ticker")["macd"].transform(lambda x: x.ewm(span=9, adjust=False).mean())

    # --- RSI (14-day) ---
    def compute_rsi(x, window=14):
        delta = x.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        avg_gain = gain.rolling(window).mean()
        avg_loss = loss.rolling(window).mean()
        rs = avg_gain / avg_loss
        return 100 - (100 / (1 + rs))
    df["rsi14"] = df.groupby("ticker")["Close"].transform(lambda x: compute_rsi(x))

    # --- Bollinger Band Width ---
    rolling_mean = df.groupby("ticker")["Close"].transform(lambda x: x.rolling(20).mean())
    rolling_std = df.groupby("ticker")["Close"].transform(lambda x: x.rolling(20).std())
    df["bb_width"] = (rolling_std * 2) / rolling_mean

    # --- Target variable (label for model training) ---
    df["target"] = (df.groupby("ticker")["Close"].shift(-1) > df["Close"]).astype(int)

    # --- Clean NaNs ---
    df = df.dropna().reset_index(drop=True)
    return df

print("⚙️ Adding technical indicators and target...")
combined_df = add_technical_indicators(combined_df)
print("✅ Indicators and target added successfully!")
print("Tickers found:", combined_df["ticker"].unique())
print("Columns:", combined_df.columns.tolist())
print(combined_df.head())


⚙️ Adding technical indicators and target...


ValueError: Cannot set a DataFrame with multiple columns to the single column ma5

In [ ]:
# ---------- Cell 6B: Save per-ticker scalers ----------
from sklearn.preprocessing import StandardScaler
import joblib
import os
# ---------- Define consistent feature list ----------
features = [
    "Close", "Open", "High", "Low", "Volume", "returns",
    "ma5", "ma10", "ma20", "ema10", "ema20",
    "momentum", "volatility", "macd_signal", "rsi14"
]
print(f"✅ Feature list set. Using {len(features)} features.")
scalers = {}
for t in combined_df["ticker"].unique():
    df_t = combined_df[combined_df["ticker"] == t]
    scaler = StandardScaler()
    scaler.fit(df_t[features])
    scalers[t] = scaler

scaler_path = os.path.join(ARTIFACT_DIR, "multi_stock_scalers.pkl")
joblib.dump(scalers, scaler_path)
print(f"✅ Saved per-ticker scalers at {scaler_path}")


In [ ]:
# ---------- Cell 7: Enhanced Model, Dataset Class & Training Loop ----------

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from tqdm import tqdm

# ---------- Dataset Class ----------
class MultiStockSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len=21):
        self.feature_cols = feature_cols
        self.seq_len = seq_len
        self.samples = []
        tickers = df['ticker'].unique()

        for t in tickers:
            df_t = df[df['ticker'] == t].sort_values('date').reset_index(drop=True)
            data = df_t[feature_cols].values.astype(np.float32)
            targets = df_t['target'].values.astype(np.float32)
            for i in range(len(df_t) - seq_len):
                x_seq = data[i:i+seq_len]
                y_val = targets[i+seq_len]
                self.samples.append((x_seq, y_val))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor([y])


# ---------- Attention BiLSTM Model ----------
class AttentionBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.fc(context)


# ---------- Focal Loss ----------
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


# ---------- Prepare Data ----------
train_dataset = MultiStockSequenceDataset(combined_df, features, seq_len=LOOKBACK_WINDOW)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

# ---------- Initialize Model ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionBiLSTM(input_dim=len(features)).to(device)
model = torch.compile(model)  # optional acceleration

criterion = FocalLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


# ---------- Training Loop ----------
EPOCHS = 15
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x)

        # Label smoothing (0.05)
        y_smooth = y * 0.9 + 0.05
        loss = criterion(preds, y_smooth)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.5f}")

# ---------- Save Model ----------
os.makedirs(ARTIFACT_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(ARTIFACT_DIR, "attention_bilstm_model.pt"))
print("✅ Model training complete and saved!")


In [ ]:
# ============================ CELL 8: FINAL FIXED INFERENCE ============================
import numpy as np
import torch
import joblib
import pandas as pd
import os

# Load per-ticker scalers
scaler_path = os.path.join(ARTIFACT_DIR, "multi_stock_scalers.pkl")
scalers = joblib.load(scaler_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

predictions = []

# Figure out how many features model expects
expected_input_size = model.lstm.input_size if hasattr(model, "lstm") else len(features)
print(f"🔍 Model expects {expected_input_size} features per timestep.\n")

for ticker in TICKERS:
    df_t = combined_df[combined_df["ticker"] == ticker].sort_values("date").reset_index(drop=True)

    if len(df_t) < LOOKBACK_WINDOW:
        print(f"⚠️ Skipping {ticker}: only {len(df_t)} rows (< LOOKBACK_WINDOW={LOOKBACK_WINDOW})")
        continue

    # ---- Preserve date and features separately ----
    last_seq_df = df_t.tail(LOOKBACK_WINDOW).copy()
    last_dates = last_seq_df["date"].tolist()  # keep for debugging
    feature_df = last_seq_df[features].copy()

    # ---- Apply ticker-specific scaler safely ----
    if ticker in scalers:
        scaler_features = list(scalers[ticker].feature_names_in_)

        # Align columns safely (fill missing, ignore extras)
        for col in scaler_features:
            if col not in feature_df.columns:
                feature_df[col] = 0.0  # add missing with neutral value
        feature_df = feature_df[scaler_features]

        try:
            scaled = scalers[ticker].transform(feature_df)
            feature_df = pd.DataFrame(scaled, columns=scaler_features)
        except Exception as e:
            print(f"⚠️ Scaling failed for {ticker}: {e}")
            continue
    else:
        print(f"⚠️ No scaler found for {ticker}, skipping scaling.")
        continue

    # ---- Ensure input feature count matches model ----
    num_features = feature_df.shape[1]
    if num_features != expected_input_size:
        print(f"⚠️ Adjusting {ticker} input size: expected {expected_input_size}, found {num_features}.")
        if num_features > expected_input_size:
            feature_df = feature_df.iloc[:, :expected_input_size]
        else:
            pad = np.zeros((LOOKBACK_WINDOW, expected_input_size - num_features))
            feature_df = np.hstack([feature_df.values, pad])
            feature_df = pd.DataFrame(feature_df)

    # ---- Predict next-day up probability ----
    X_last = torch.tensor(feature_df.values, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(X_last)
        prob_up = torch.sigmoid(output).cpu().item()
        signal = "UP" if prob_up >= 0.5 else "DOWN"

    predictions.append({
        "ticker": ticker,
        "predicted_prob_up": round(prob_up, 4),
        "signal": signal,
        "last_date": last_dates[-1]
    })

# ---- Display results ----
pred_df = pd.DataFrame(predictions)
print("\n✅ Inference complete!\n")
print(pred_df)


In [ ]:
# ---------- Next-day predictions for all tickers ----------
threshold = 0.5  # probability cutoff to classify up/down

predictions = []

for ticker in TICKERS:
    df_t = combined_df[combined_df['ticker']==ticker].sort_values('date').reset_index(drop=True)

    # skip if not enough rows
    if len(df_t) < LOOKBACK_WINDOW:
        print(f"Not enough data for {ticker}, skipping...")
        continue

    last_seq = df_t[features].iloc[-LOOKBACK_WINDOW:].values.astype(np.float32)

    # create tensor
    x = torch.tensor(last_seq).unsqueeze(0).to(device)

    # model prediction
    model.eval()
    with torch.no_grad():
        p_up = model(x).cpu().numpy().flatten()[0]

    # classify
    signal = "UP" if p_up > threshold else "DOWN"

    predictions.append({
        "ticker": ticker,
        "prob_up": p_up,
        "signal": signal
    })

# display results
import pandas as pd
pred_df = pd.DataFrame(predictions)
pred_df = pred_df.sort_values('prob_up', ascending=False).reset_index(drop=True)
print(pred_df)


In [ ]:
from datetime import datetime

# Ensure directory exists
PRED_DIR = os.path.join(ARTIFACT_DIR, "predictions")
os.makedirs(PRED_DIR, exist_ok=True)

# File path for today's predictions
today_str = datetime.now().strftime("%Y-%m-%d")
pred_file = os.path.join(PRED_DIR, f"predictions_{today_str}.csv")

# ---------- Generate predictions ----------
threshold = 0.5
predictions = []

for ticker in TICKERS:
    df_t = combined_df[combined_df['ticker']==ticker].sort_values('date').reset_index(drop=True)

    if len(df_t) < LOOKBACK_WINDOW:
        print(f"Not enough data for {ticker}, skipping...")
        continue

    last_seq = df_t[features].iloc[-LOOKBACK_WINDOW:].values.astype(np.float32)

    x = torch.tensor(last_seq).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        p_up = model(x).cpu().numpy().flatten()[0]

    signal = "UP" if p_up > threshold else "DOWN"

    predictions.append({
        "timestamp": datetime.now(),
        "ticker": ticker,
        "prob_up": p_up,
        "signal": signal
    })

# Convert to DataFrame
pred_df = pd.DataFrame(predictions)

# Save to CSV (append if file exists)
if os.path.exists(pred_file):
    pred_df.to_csv(pred_file, mode='a', index=False, header=False)
else:
    pred_df.to_csv(pred_file, index=False)

print(f"✅ Predictions saved to {pred_file}")
print(pred_df)


In [ ]:
import glob

# ---------- Load all saved prediction files ----------
pred_files = sorted(glob.glob(os.path.join(PRED_DIR, "predictions_*.csv")))

all_preds = []
for f in pred_files:
    df = pd.read_csv(f, parse_dates=['timestamp'])
    all_preds.append(df)

if not all_preds:
    print("No prediction files found.")
else:
    all_preds_df = pd.concat(all_preds, ignore_index=True)
    print(f"Loaded {len(all_preds_df)} total predictions.")

# ---------- Merge with actual next-day returns ----------
actuals = []

for ticker in TICKERS:
    df_t = combined_df[combined_df['ticker']==ticker].sort_values('date').reset_index(drop=True)
    df_t['next_day_target'] = df_t['target'].shift(-1)  # 1 if next day up, 0 if down
    df_t = df_t[['date','next_day_target']].iloc[-len(all_preds_df):]  # align length
    df_t['ticker'] = ticker
    actuals.append(df_t)

actuals_df = pd.concat(actuals, ignore_index=True)

# Merge predictions with actuals
merged_df = pd.merge(all_preds_df, actuals_df, how='left', left_on=['ticker'], right_on=['ticker'])
merged_df['correct'] = (merged_df['signal'] == merged_df['next_day_target'].map({1.0:'UP', 0.0:'DOWN'}))

# Overall accuracy
accuracy = merged_df['correct'].mean()
print(f"📊 Overall backtest accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Display last few predictions with results
merged_df.sort_values('timestamp', ascending=False).head(10)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure merged_df exists from Cell 11
if 'merged_df' not in globals():
    raise ValueError("Run backtesting cell first!")

# ---------- 1️⃣ Overall accuracy ----------
overall_acc = merged_df['correct'].mean()
print(f"📊 Overall accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")

# ---------- 2️⃣ Accuracy per ticker ----------
ticker_acc = merged_df.groupby('ticker')['correct'].mean().reset_index()
plt.figure(figsize=(8,4))
sns.barplot(data=ticker_acc, x='ticker', y='correct', palette='viridis')
plt.title("✅ Accuracy per Ticker")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

# ---------- 3️⃣ Probability distribution ----------
plt.figure(figsize=(10,5))
sns.histplot(data=merged_df, x='prob_up', hue='ticker', bins=20, kde=True, palette='magma', alpha=0.6)
plt.title("📈 Predicted Probability Distribution (Next-day Up)")
plt.xlabel("Probability of Up")
plt.ylabel("Count")
plt.show()

# ---------- 4️⃣ Cumulative returns simulation ----------
# simulate +1 if signal correct, -1 if wrong
merged_df['sim_return'] = merged_df['correct'].map({True: 1, False: -1})
merged_df['cumulative_return'] = merged_df.groupby('ticker')['sim_return'].cumsum()

plt.figure(figsize=(10,5))
for ticker in TICKERS:
    df_t = merged_df[merged_df['ticker']==ticker]
    plt.plot(df_t['timestamp'], df_t['cumulative_return'], label=ticker)
plt.title("💹 Simulated Cumulative Performance by Ticker")
plt.xlabel("Timestamp")
plt.ylabel("Cumulative Return (Simulation)")
plt.legend()
plt.show()


In [ ]:
!pip install gradio --quiet


In [ ]:
import gradio as gr
import torch
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

# Load model & scalers
model.eval()
scalers = joblib.load(os.path.join(ARTIFACT_DIR, "multi_stock_scalers.pkl"))

def predict_ticker(ticker):
    df_t = combined_df[combined_df['ticker']==ticker].sort_values('date').reset_index(drop=True)
    last_seq = df_t[features].iloc[-LOOKBACK_WINDOW:].values.astype(np.float32)
    x = torch.tensor(last_seq).unsqueeze(0).to(device)

    with torch.no_grad():
        p_up = model(x).cpu().numpy().flatten()[0]

    # Get prediction timestamp
    prediction_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    last_data_date = df_t['date'].iloc[-1]

    signal = "📈 UP" if p_up > 0.5 else "📉 DOWN"
    confidence = max(p_up, 1 - p_up) * 100

    # Create detailed result
    result = f"""
    ### Prediction Summary

    **Ticker:** {ticker}
    **Signal:** {signal}
    **Confidence:** {confidence:.1f}%
    **Probability (Up):** {p_up:.3f}
    **Probability (Down):** {1-p_up:.3f}

    ---

    **Prediction Time:** {prediction_time}
    **Model Window:** {LOOKBACK_WINDOW} days
    """

    # Create chart data
    chart_data = pd.DataFrame({
        'Direction': ['Up', 'Down'],
        'Probability': [p_up * 100, (1 - p_up) * 100]
    })

    return result, chart_data

# Enhanced Gradio Interface with custom theme
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 📊 Stock Market Prediction Dashboard
        This tool uses a BiLSTM deep learning model to predict whether a stock will move **up** or **down** the next trading day.
        Select a ticker below to generate a real-time prediction.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            ticker_input = gr.Dropdown(
                choices=TICKERS,
                label="🎯 Select Stock Ticker",
                info="Choose from available tickers",
                value=TICKERS[0] if TICKERS else None
            )
            predict_btn = gr.Button("🔮 Generate Prediction", variant="primary", size="lg")

            gr.Markdown(
                """
                ---
                ### ℹ️ How to Use
                1. Select a stock ticker from the dropdown
                2. Click "Generate Prediction"
                3. View the prediction confidence and probabilities

                **Note:** Predictions are based on historical patterns and should not be used as sole investment advice.
                """
            )

        with gr.Column(scale=2):
            result_output = gr.Markdown(label="Prediction Results")
            chart_output = gr.BarPlot(
                x="Direction",
                y="Probability",
                title="Probability Distribution",
                y_title="Probability (%)",
                height=300,
                width=500
            )

    predict_btn.click(
        fn=predict_ticker,
        inputs=ticker_input,
        outputs=[result_output, chart_output]
    )

    # Auto-predict on dropdown change (optional)
    ticker_input.change(
        fn=predict_ticker,
        inputs=ticker_input,
        outputs=[result_output, chart_output]
    )

    gr.Markdown(
        """
        ---
        <div style='text-align: center; color: #666; font-size: 0.9em;'>
        <p><strong>Disclaimer:</strong> This model is for educational purposes only. Past performance does not guarantee future results.</p>
        <p>Always conduct thorough research and consult with financial advisors before making investment decisions.</p>
        </div>
        """
    )

demo.launch(share=True)

In [ ]:
# --- Debug Cell: check target balance ---
combined_df['target'].value_counts(normalize=True)


In [ ]:
# 🔍 Check what tickers exist and what features are present
print("Unique tickers in combined_df:", combined_df['ticker'].unique())
print("\nFeature columns:", [c for c in combined_df.columns if c not in ['ticker','date','target']])

# Check dataset sizes
for t in TICKERS:
    print(f"{t}: {len(combined_df[combined_df['ticker']==t])} rows")

# Check scaler feature names
for t in scalers.keys():
    print(f"{t} scaler features:", scalers[t].feature_names_in_)
